In [1]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
# ============================================================
# PATHS
# ============================================================

TCGA_PATH = "/data/kumarr17/merged_sample_mtx/TCGA_Merged_mRNA_Expression.tsv"

FEATURE_PATH = "/data/kumarr17/common_features_ver3.txt"


# ============================================================
# LOAD TRAINING DATA + FIXED FEATURES
# ============================================================

tcga_features_list = list(
    pd.read_csv(FEATURE_PATH, header=None)[0]
)

df_tcga = pd.read_csv(
    TCGA_PATH,
    index_col=0
)


# ============================================================
# FCNN MODEL
# ============================================================

def build_fcnn(input_dim):

    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="mean_squared_error",
        metrics=["mae"]
    )

    return model


# ============================================================
# TXCYTO PREDICTION FUNCTION
# ============================================================

def predict_txcyto(
    test_df,
    cytokines,
    epochs=50,
    batch_size=128,
    verbose=0
):
# ============================================================
# REPRODUCIBILITY
# ============================================================
    tf.keras.backend.clear_session()
    SEED = 42
    
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    # --------------------------------------------------------
    # Cytokine input
    # --------------------------------------------------------

    if isinstance(cytokines, str):
        cytokines = [cytokines]

    cytokines = list(cytokines)
    
    # --------------------------------------------------------
    # Check requested cytokines
    # --------------------------------------------------------
    
    available_cytokines = [
        c for c in cytokines
        if c in df_tcga.columns
    ]
    
    missing_cytokines = [
        c for c in cytokines
        if c not in df_tcga.columns
    ]
    
    
    # Report unavailable cytokines
    if missing_cytokines:
        print(
            "\nWarning: The following cytokines are not available "
            "in TxCyto and will be skipped:"
        )
        print(missing_cytokines)
    
    
    # Stop only if NONE of the requested cytokines are available
    if len(available_cytokines) == 0:
        raise ValueError(
            "None of the requested cytokines are available in TxCyto."
        )
    
    
    print(
        f"\nPredicting {len(available_cytokines)} "
        f"of {len(cytokines)} requested cytokines:"
    )
    
    print(available_cytokines)
    
    
    # Use only available cytokines from this point onward
    cytokines = available_cytokines    
    # --------------------------------------------------------
    # Check test features
    # --------------------------------------------------------

    missing_features = [
        gene for gene in tcga_features_list
        if gene not in test_df.columns
    ]

    if len(missing_features) > 0:

        raise ValueError(
            f"{len(missing_features)} required TxCyto features "
            "are missing from the test dataset.\n"
            f"First missing genes: {missing_features[:20]}"
        )


    # --------------------------------------------------------
    # Select features in EXACT training order
    # --------------------------------------------------------

    df_train_features = df_tcga[tcga_features_list].copy()

    df_test_features = test_df[tcga_features_list].copy()


    # --------------------------------------------------------
    # Convert to float32
    # --------------------------------------------------------

    X_train_raw = (
        df_train_features
        .values
        .astype("float32")
    )

    X_test_raw = (
        df_test_features
        .values
        .astype("float32")
    )


    # --------------------------------------------------------
    # Row-wise z-score
    #
    # IMPORTANT:
    # Each sample is normalized across genes independently.
    # --------------------------------------------------------

    X_train_raw = (
        X_train_raw -
        X_train_raw.mean(axis=1, keepdims=True)
    ) / (
        X_train_raw.std(axis=1, keepdims=True)
        + 1e-8
    )


    X_test_raw = (
        X_test_raw -
        X_test_raw.mean(axis=1, keepdims=True)
    ) / (
        X_test_raw.std(axis=1, keepdims=True)
        + 1e-8
    )


    # --------------------------------------------------------
    # StandardScaler
    #
    # Fit ONLY on TCGA training samples.
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        X_train_raw
    ).astype("float32")

    X_test = scaler.transform(
        X_test_raw
    ).astype("float32")


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    early_stop = keras.callbacks.EarlyStopping(
        monitor="loss",
        patience=5,
        restore_best_weights=True
    )


    # --------------------------------------------------------
    # Train cytokine-specific models
    # --------------------------------------------------------

    pred_df_list = []


    for cytokine in cytokines:

        print("\n======================================")
        print(f"Training TxCyto model: {cytokine}")
        print("======================================")


        # ----------------------------------------------------
        # Target
        # ----------------------------------------------------

        y_train = (
            df_tcga[[cytokine]]
            .values
            .astype("float32")
        )

        # ----------------------------------------------------
        # Build model
        # ----------------------------------------------------

        model = build_fcnn(
            X_train.shape[1]
        )


        # ----------------------------------------------------
        # Train model
        # ----------------------------------------------------

        model.fit(
            X_train,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            verbose=verbose,
            callbacks=[early_stop],
            shuffle=True
        )


        # ----------------------------------------------------
        # Predict cytokine activity
        # ----------------------------------------------------

        y_pred = model.predict(
            X_test,
            verbose=0
        ).ravel()


        # ----------------------------------------------------
        # Save predictions
        # ----------------------------------------------------

        tmp = pd.DataFrame({

            "sample":
                df_test_features.index,

            "cytokine":
                cytokine,

            "predicted_activity":
                y_pred

        })


        pred_df_list.append(tmp)


    # ========================================================
    # COMBINE RESULTS
    # ========================================================

    pred_long = pd.concat(
        pred_df_list,
        axis=0,
        ignore_index=True
    )


    # --------------------------------------------------------
    # Sample × cytokine matrix
    # --------------------------------------------------------

    pred_matrix = pred_long.pivot(
        index="sample",
        columns="cytokine",
        values="predicted_activity"
    )


    print("\nTxCyto prediction completed.")
    print(f"Samples: {pred_matrix.shape[0]}")
    print(f"Cytokines: {pred_matrix.shape[1]}")
    print(f"Features used: {len(tcga_features_list)}")


    return pred_long, pred_matrix

In [2]:
#### Test gene-expression matrix sample* genes
test_df=pd.read_csv("/data/kumarr17/figures_to_upload/supplementray_table/test_gene_exp_df.csv",index_col='Unnamed: 0') 
### cytokine of interest to be predicted
#cyt_tmp22 = ["CCL26", "IL16","ABC"]
cyt_tmp22 = ["CCL26", "IL16", "IL1B", "IL13", "IL7",
    "CCL13", "CCL2", "IL12A", "IL15", "VEGFA",
    "CCL4", "IL8", "TNF", "IL1A", "CXCL10",
    "IL10", "IL6", "LTA", "CCL22", "CCL3",
    "IFNG", "CCL17", "CCL11","ABC"]
### Calling function
pred_long, pred_matrix = predict_txcyto(
    test_df=test_df,
    cytokines=cyt_tmp22
)


['ABC']

Predicting 23 of 24 requested cytokines:
['CCL26', 'IL16', 'IL1B', 'IL13', 'IL7', 'CCL13', 'CCL2', 'IL12A', 'IL15', 'VEGFA', 'CCL4', 'IL8', 'TNF', 'IL1A', 'CXCL10', 'IL10', 'IL6', 'LTA', 'CCL22', 'CCL3', 'IFNG', 'CCL17', 'CCL11']

Training TxCyto model: CCL26


2026-10-02 15:55:18.069681: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-10-02 15:55:18.069718: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2026-10-02 15:55:18.069747: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (cn0132): /proc/driver/nvidia/version does not exist
2026-10-02 15:55:18.070230: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.



Training TxCyto model: IL16

Training TxCyto model: IL1B

Training TxCyto model: IL13

Training TxCyto model: IL7

Training TxCyto model: CCL13

Training TxCyto model: CCL2

Training TxCyto model: IL12A

Training TxCyto model: IL15

Training TxCyto model: VEGFA

Training TxCyto model: CCL4

Training TxCyto model: IL8

Training TxCyto model: TNF

Training TxCyto model: IL1A

Training TxCyto model: CXCL10

Training TxCyto model: IL10

Training TxCyto model: IL6

Training TxCyto model: LTA

Training TxCyto model: CCL22

Training TxCyto model: CCL3

Training TxCyto model: IFNG

Training TxCyto model: CCL17

Training TxCyto model: CCL11

TxCyto prediction completed.
Samples: 35
Cytokines: 23
Features used: 6075


In [3]:
pred_matrix

cytokine,CCL11,CCL13,CCL17,CCL2,CCL22,CCL26,CCL3,CCL4,CXCL10,IFNG,...,IL15,IL16,IL1A,IL1B,IL6,IL7,IL8,LTA,TNF,VEGFA
sample,,,,,,,,,,,,,,,,,,,,,
G10,8.702123,106.816048,-7.709315,4642.360840,35.610603,35.003452,255.435211,475.124817,2038.195679,2.337548,...,333.666595,754.032166,-29.938974,192.246735,1309.761108,197.493454,478.566437,8.912667,55.862465,5408.141602
G102,-0.458240,572.064941,1.611257,1887.305908,-11.668349,10.684193,246.622849,472.358490,1474.050171,3.127249,...,520.837219,1784.167603,-47.154858,175.017242,263.391205,366.995209,351.253235,15.370919,89.720367,14137.421875
G11,22.206310,101.677864,5.534124,7270.450684,18.295961,13.405155,207.231766,119.705742,993.385010,3.409609,...,283.611237,479.863647,-93.226677,338.800812,2100.468018,276.449158,684.522766,0.444646,48.590431,5830.484375
G124,56.633076,57.399830,-16.784082,1345.537598,-11.693362,14.694859,102.800049,57.264503,1540.639893,8.092755,...,164.393051,330.272156,-54.240570,202.583282,84.288086,106.441658,772.685486,1.325490,66.254349,3886.246094
G136,32.634872,75.978409,-29.833277,1288.375122,10.985353,-8.143475,155.338638,145.416489,2431.283691,15.897490,...,415.345612,722.929993,-72.231049,45.550377,36.763012,92.264114,543.221985,41.179279,41.786392,3351.536621
G138,64.915726,29.958340,50.459812,3695.583740,29.715361,30.607853,213.417847,247.378922,1533.088989,11.010277,...,490.743073,2877.490479,-14.688517,381.297089,727.930725,141.183838,575.931396,40.854897,9.711415,4096.230957
G144,16.198933,141.330017,-16.665903,4638.967285,-20.118376,28.565525,114.080437,166.577988,1614.299316,-5.435273,...,343.852356,1354.267578,-22.480602,113.066582,510.000092,245.613068,428.032806,2.274417,45.286774,699.130493
G15,30.083952,133.668777,8.128637,1812.164551,-17.705315,32.739994,286.150269,223.597443,1326.444946,27.488392,...,472.606812,903.427124,-28.912516,193.449921,190.103897,152.935287,469.913055,27.102846,43.340843,4981.217285
G17,36.010632,117.501091,42.424366,3824.260254,48.849133,25.422367,239.339752,283.565674,1218.167114,0.852374,...,291.056976,1027.892090,-97.425156,316.770416,695.849854,233.232407,609.638123,1.128762,45.434601,3791.476074
